# Create Dave config

Builds three independent Dave recipe XMLs from `round_info.csv` and the
Kilroy config for `MICROSCOPE`, in acquisition order: focus-lock test,
cells, hybs. Each section's printed cost estimate assumes it starts right
after the previous one finishes -- see `FOCUS_TEST_LEAD_MIN` below.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave   import (
    create_dave_config, dave_config_filename, dave_cells_config_filename,
    dave_focustest_config_filename, create_focus_test_dave_config,
)
from MERci.acquisition.kilroy import find_kilroy_config
from MERci.acquisition.display import display_xml
from MERci.analysis.stage_z   import read_off_file_if_ready, summarize_focus_lock

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# Positions files are named with POSITIONS_TAG, Dave recipes with SAMPLE_NAME.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "ST2"   # same as notebooks 01/05
USE_ADAPTORS         = False    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = True   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = False    # True = first hyb (after the cells round) omits the cleave step

# Imaging round 1 = cells; rounds 2..N_HYBS+1 = bits 1..N. The first hyb's fluidics has no cleave.

# ── Read what notebook 03 produced ──────────────────────────────────
round_info_path = METADATA_DIR / "round_info.csv"
if not round_info_path.exists():
    raise FileNotFoundError(f"{round_info_path} not found -- run notebook 05 first.")
round_info = pd.read_csv(round_info_path)

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(f"{rbc_path} not found -- run notebook 05 first.")
N_HYBS = int(pd.read_csv(rbc_path)["round"].max())

MULTI_BOUNDARY = "positions_file" in round_info.columns
print(f"N_HYBS               : {N_HYBS}")
print(f"Multi-boundary layout: {MULTI_BOUNDARY}")
print(f"Use adaptors         : {USE_ADAPTORS}")
print(f"Final cleave         : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave  : {FIRST_HYB_NO_CLEAVE}")

In [ ]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 03): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# The Dave recipe's fluidic steps use this Kilroy config's protocol names, so
# every one exists in Kilroy. Microscopes without one use MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

# Cells is imaging round 1; every later round is a hyb round.
cells_round_info = round_info[round_info["imaging_round"] == 1]
hybs_round_info  = round_info[round_info["imaging_round"] > 1]

## Focus-lock test recipe

A separate Dave recipe, run first: visits every FOV and checks the focus lock,
with no fluidics, to catch a bad lock before the full acquisition.

`FOCUS_TEST_LEAD_MIN` is when it is expected to start; the cells and hybs
start times follow from it.

`N_TEST_FRAMES`:
- `0` (default) -- check focus only, no image. Nothing is saved per FOV:
  watch Dave's warnings panel while the recipe runs.
- `> 0` -- also takes a short movie per FOV, with notebook 01's focus-test HAL
  config. Each movie's `.off` file records the lock per frame
  (`good-offset`); the last cell reads them back.

In [ ]:
# ── Focus-lock test parameters ──────────────────────────────────────────
FOCUS_TEST_LEAD_MIN = 30    # minutes until this recipe starts; cells/hybs follow
N_TEST_FRAMES    = 0        # 0 = check focus only; > 0 = frames per FOV (writes .off files)
NUM_FOCUS_CHECKS = 50
FOCUS_SCAN       = True

# Focus-test HAL config from notebook 01.
_focustest_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*-focustest-*.xml"))
if not _focustest_hal_configs:
    raise FileNotFoundError(
        f"No focus-test HAL config found in {SETTINGS_DIR} -- run notebook 01's "
        f"'create hal config for focus test' section first."
    )
TEST_HAL_CONFIG = _focustest_hal_configs[-1].name
print(f"Focus-test HAL config: {TEST_HAL_CONFIG}")

# Single-positions layout only. For a multi-tissue layout, set
# positions_file by hand.
focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
if not focus_test_positions.exists():
    raise FileNotFoundError(f"{focus_test_positions} not found -- run notebook 03 first.")

focus_test_output   = SETTINGS_DIR / dave_focustest_config_filename(MICROSCOPE, SAMPLE_NAME)
focus_test_data_dir = (SAMPLE_DIR / "data" / "focus_test") if N_TEST_FRAMES > 0 else None

focus_test_start = datetime.now() + timedelta(minutes=FOCUS_TEST_LEAD_MIN)

n_fovs, focustest_est = create_focus_test_dave_config(
    positions_file   = focus_test_positions,
    output_path      = focus_test_output,
    num_focus_checks = NUM_FOCUS_CHECKS,
    focus_scan       = FOCUS_SCAN,
    n_test_frames    = N_TEST_FRAMES,
    hal_config       = TEST_HAL_CONFIG,
    settings_dir     = SETTINGS_DIR,
    data_dir         = focus_test_data_dir,
    movie_name       = f"hal-{MICROSCOPE.lower()}-focustest",
    kilroy_config    = KILROY_CONFIG,
    microscope       = MICROSCOPE,
    start_time       = focus_test_start,
)

print(f"Focus-lock test recipe saved: {focus_test_output}")
print(f"FOVs visited: {n_fovs}")
if N_TEST_FRAMES > 0:
    print(f"Each FOV takes a real {N_TEST_FRAMES}-frame movie -> .off files land under {focus_test_data_dir}")
else:
    print("Check-focus only -- no movie taken, no persisted per-FOV record (see markdown above).")

display_xml(focus_test_output)

In [ ]:
# ── Read back per-FOV focus-lock results ──────────────────────────────────
# Needs N_TEST_FRAMES > 0 and the recipe run on the microscope.
if N_TEST_FRAMES > 0:
    # Dave zero-pads the FOV index to the digits of the FOV count.
    pad = len(str(n_fovs))
    bad_fovs  = []
    n_checked = 0
    for fov_idx in range(n_fovs):
        off_path = focus_test_data_dir / f"hal-{MICROSCOPE.lower()}-focustest_{str(fov_idx).zfill(pad)}.off"
        # None = not written yet, or still being written.
        off_df = read_off_file_if_ready(off_path)
        if off_df is None:
            continue
        n_checked += 1
        summary = summarize_focus_lock(off_df)
        if not summary["all_good"]:
            bad_fovs.append((fov_idx, summary))

    print(f"Checked {n_checked} / {n_fovs} FOV(s) (missing ones haven't been imaged yet).")
    if bad_fovs:
        print(f"{len(bad_fovs)} FOV(s) with a bad focus lock:")
        for fov_idx, summary in bad_fovs:
            print(f"  FOV {fov_idx}: {summary['n_bad_frames']}/{summary['n_frames']} bad frame(s)")
    elif n_checked:
        print("All checked FOVs had a good focus lock.")
    else:
        print("No FOVs found yet -- run the recipe on the microscope first.")
else:
    print("N_TEST_FRAMES == 0 -- no per-FOV file was written (see markdown above); "
          "check Dave's own warnings panel while the recipe runs instead.")

## Cells

The cells-only Dave recipe: one imaging round, no fluidics (bit 1's
hybridization starts the hybs recipe).

In [ ]:
# ── Build the cells recipe ────────────────────────────────────────────────
cells_output = SETTINGS_DIR / dave_cells_config_filename(MICROSCOPE, SAMPLE_NAME)

# Assumed to start right after the focus-lock test recipe above finishes.
cells_start = focus_test_start + timedelta(seconds=focustest_est.total_time_s)

cells_est = create_dave_config(
    round_info            = cells_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = cells_output,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    include_final_cleave  = False,   # no fluidics belongs in the cells-only file
    microscope            = MICROSCOPE,
    start_time            = cells_start,
)

print(f"Cells Dave config saved: {cells_output}")
display_xml(cells_output)

## Hybs

The hybs Dave recipe: every hyb round and its fluidics, starting with the
fluidics before hyb 1, so it runs on its own.

In [ ]:
# ── Build the hybs recipe ─────────────────────────────────────────────────
dave_output = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

# Assumed to start right after the cells recipe above finishes.
hybs_start = cells_start + timedelta(seconds=cells_est.total_time_s)

hybs_est = create_dave_config(
    round_info            = hybs_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = dave_output,
    use_adaptors          = USE_ADAPTORS,
    include_final_cleave  = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave   = FIRST_HYB_NO_CLEAVE,
    leading_fluidics      = True,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    microscope            = MICROSCOPE,
    start_time            = hybs_start,
)

print(f"Hybs Dave config saved: {dave_output}")
display_xml(dave_output)